# 03 Compuertas de calidad y deduplicación

Bronze conserva los mensajes. Silver exige MMSI de nueve dígitos distinto de cero, tiempo válido, coordenadas finitas en rango, posición distinta de (0, 0), SOG en [0, 60] nudos y COG en [0, 360) grados. `COG = 360` es el centinela de rumbo no disponible. Heading, eslora y manga pueden faltar. Una fila inválida se conserva con todos sus motivos.

A propósito no hay chequeo de tierra contra mar. El dato NOAA incluye puertos, ríos y los Grandes Lagos, y un mapa de tierra grueso marcaría tráfico real como si estuviera en tierra.

**Por qué MMSI y BaseDateTime.** El MMSI identifica el emisor y el tiempo UTC identifica su observación. Agregar latitud o longitud a la llave retendría dos posiciones incompatibles para el mismo instante y distorsionaría la distancia en oro. Usar solo MMSI descartaría la trayectoria. Primero se evalúa cada copia para que una copia inválida no desplace a la válida. Entre candidatas válidas del mismo batch se elige de manera determinista por archivo y contenido. Si la clave ya fue aceptada en un batch anterior, se conserva la primera versión.

Cada batch reconcilia entradas con nuevas, duplicadas dentro del batch, duplicadas anteriores y cuarentena. El checkpoint `silver_ais_v2` relee el histórico de Bronze la primera vez.

| motivo | Condición | Por qué |
|---|---|---|
| `mmsi_invalido` | `MMSI` nulo, no tiene nueve dígitos, o es `000000000` | Sin buque no hay trayectoria. |
| `sin_timestamp` | `BaseDateTime` nulo | Sin instante no hay `fecha_dia` ni clave del `MERGE`. |
| `latitud_invalida` | `LAT` nula, NaN, o fuera de [-90, 90] | No es una posición para haversine ni para H3. |
| `longitud_invalida` | `LON` nula, NaN, o fuera de [-180, 180] | Igual que la latitud. |
| `posicion_cero` | `LAT` y `LON` son exactamente 0 | El par (0, 0) es un centinela, no una posición. Un cero en un solo eje no basta: el ecuador y el meridiano de Greenwich son coordenadas reales. |
| `sog_invalido` | `SOG` nulo, NaN, o fuera de [0, 60] | Incluye el centinela ITU `102.3`. Gold calcula `sog_media` con estas filas. |
| `cog_invalido` | `COG` nulo, NaN, o fuera de [0, 360) | `COG = 360` es rumbo no disponible y va a cuarentena. |

`Heading = 511` se queda, con su valor. Es rumbo ausente, no una posición mala.


In [ ]:
# Capturar los parámetros del notebook llamador antes de %run.
dbutils.widgets.text("fecha", "2023-06-08")
dbutils.widgets.text("ancla_replay", "")
dbutils.widgets.text("inicio_run", "")
ow_argumentos = tuple(dbutils.widgets.get(k).strip() for k in ("fecha", "ancla_replay", "inicio_run"))

In [ ]:
%run ./_config

In [ ]:
from pyspark.sql import Window, functions as F

COLUMNAS = ["MMSI", "BaseDateTime", "LAT", "LON", "SOG", "COG", "Heading", "VesselName",
            "IMO", "CallSign", "VesselType", "Status", "Length", "Width", "Draft", "Cargo",
            "TransceiverClass", "_source_file", "_ingested_at"]


def con_motivo(df):
    lat, lon = F.col("LAT"), F.col("LON")
    lat_ok = lat.between(-90, 90) & ~F.isnan(lat)
    lon_ok = lon.between(-180, 180) & ~F.isnan(lon)
    mmsi = F.col("MMSI")
    return df.select(*COLUMNAS, F.concat_ws(",",
        F.when(mmsi.isNull() | ~mmsi.rlike("^[0-9]{9}$") | (mmsi == "000000000"), "mmsi_invalido"),
        F.when(F.col("BaseDateTime").isNull(), "sin_timestamp"),
        F.when(lat.isNull() | ~lat_ok, "latitud_invalida"),
        F.when(lon.isNull() | ~lon_ok, "longitud_invalida"),
        F.when((lat == 0) & (lon == 0), "posicion_cero"),
        F.when(F.col("SOG").isNull() | ~F.col("SOG").between(0, 60) | F.isnan("SOG"), "sog_invalido"),
        F.when(F.col("COG").isNull() | ~F.col("COG").between(0, 360) |
               (F.col("COG") == 360) | F.isnan("COG"), "cog_invalido"),
    ).alias("motivo"))


def escribir_batch(df, tabla, app, batch_id):
    (df.write.format("delta").mode("append")
     .option("txnAppId", app).option("txnVersion", int(batch_id)).saveAsTable(tabla))


def aplicar_silver(batch_df, batch_id):
    sesion = batch_df.sparkSession
    marcado = con_motivo(batch_df).persist()
    candidatas = None
    try:
        tot = marcado.agg(F.count("*"), F.count(F.when(F.col("motivo") != "", 1))).first()
        entrada, rechazo = tot[0], tot[1]
        if entrada == 0:
            return
        app = "oceanwatch-v2-" + SILVER.replace("`", "")
        if rechazo:
            malas = marcado.where("motivo <> ''").withColumn("batch_calidad", F.lit(int(batch_id)).cast("long"))
            escribir_batch(malas, CUARENTENA, app + "-rechazos", batch_id)
        validas = marcado.where("motivo = ''").drop("motivo")
        orden = Window.partitionBy("MMSI", "BaseDateTime").orderBy(
            "_source_file", F.to_json(F.struct(*COLUMNAS[:-2])), "_ingested_at")
        candidatas = (validas.withColumn("_orden", F.row_number().over(orden)).where("_orden = 1")
                      .drop("_orden").withColumn("fecha_dia", F.to_date("BaseDateTime"))
                      .withColumn("batch_alta", F.lit(int(batch_id)).cast("long")).persist())
        dias = candidatas.groupBy("fecha_dia").count().collect()
        n = sum(x["count"] for x in dias)
        nuevas = 0
        if dias:
            fechas = ", ".join("DATE '" + x.fecha_dia.isoformat() + "'" for x in dias)
            candidatas.createOrReplaceTempView("ow_candidatas")
            sesion.sql(f"""MERGE INTO {SILVER} d USING ow_candidatas s
                ON d.fecha_dia IN ({fechas}) AND d.fecha_dia=s.fecha_dia
                  AND d.MMSI=s.MMSI AND d.BaseDateTime=s.BaseDateTime
                WHEN NOT MATCHED THEN INSERT *""")
            nuevas = sesion.sql(
                f"SELECT count(*) FROM {SILVER} WHERE fecha_dia IN ({fechas}) AND batch_alta={int(batch_id)}"
            ).first()[0]
        intra, previos = entrada - rechazo - n, n - nuevas
        if min(intra, previos) < 0 or entrada != nuevas + intra + previos + rechazo:
            raise ValueError(
                f"Balance del batch {batch_id}: entrada={entrada} nuevas={nuevas} "
                f"duplicados_batch={intra} duplicados_previos={previos} cuarentena={rechazo}"
            )
        metricas = sesion.sql(f"""SELECT {int(batch_id)}L batch, DATE '{fecha}' fecha_job,
            {entrada}L entrada, {nuevas}L nuevas, {intra}L duplicados_batch,
            {previos}L duplicados_previos, {rechazo}L cuarentena, current_timestamp() ejecutado_en""")
        escribir_batch(metricas, METRICAS_V2, app + "-metricas", batch_id)
    finally:
        if candidatas is not None:
            candidatas.unpersist()
        marcado.unpersist()


## Ejecución

El stream lee bronze con el checkpoint `silver_ais_v2`. La primera ejecución relee el histórico. Las siguientes solo ven lo que bronze aún no había entregado.


In [ ]:
consulta = (
    spark.readStream.format("delta").table(BRONZE)
    .writeStream.foreachBatch(aplicar_silver)
    .queryName("ow_silver_v2")
    .option("checkpointLocation", CHECKPOINT_SILVER)
    .trigger(availableNow=True)
    .start()
)
consulta.awaitTermination()
